In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#!/usr/bin/env python
# coding: utf-8
import sys
sys.path.append("/work/public_project/")

import json
import os
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.configs.parameter_converter import dict_to_config
from src.evaluation.evaluate_result import evaluate_performance, print_evaluation_summary
from src.io.treat_yaml import load_yaml, save_yaml
from src.preprocess.type_registry_builder import TypeSystem, append_colors_to_registry, build_type_registry
from src.preprocess.celltype_to_cluster import load_color_info, plot_color_palette
from src.preprocess.gene_registry import GeneRegistry
from src.steps.step1.run import run_step1
from src.steps.step1.visualize import visualize_step1
from src.steps.step2.run import run_step2
from src.steps.step2.visualize import visualize_step2
from src.io.save_each_phase import save_step1_outputs, save_step2_outputs, save_eval_outputs

In [ ]:
base_config_path = Path(
    "/work/public_project/experiments/config/config_xenium_human_lung_cancer.yaml"
)
raw_cfg = load_yaml(base_config_path)
cfg = dict_to_config(raw_cfg)


save_dir = Path(f"/work/public_project/experiments/runs")
dataset_dir = save_dir / f"{cfg.base_config.measurement_type}_{cfg.base_config.tissue_type}"
os.makedirs(dataset_dir, exist_ok=True)
print("Save directory:", dataset_dir)

if (dataset_dir / "ts_data.yaml").exists():
    ts_data = load_yaml(dataset_dir / "ts_data.yaml")
else:
    ts_data = {
        "ts_celltype_estimation": None,
        "ts_cell_estimation": None,
        "Notes": None,
    }
    save_yaml(ts_data, dataset_dir / "ts_data.yaml")

# データの読み込み
gene_df = pd.read_parquet('/work/public_project/experiments/data/sample_st_data.parquet', engine='pyarrow')
MarkerDict = dict[str, dict[str, list[str]]]
def load_marker_dict(path: str | Path) -> MarkerDict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

markers = load_marker_dict("/work/public_project/experiments/data/samlple_markers.json")

gene_col = "gene"
st_x_col = "local_pixel_x"
st_y_col = "local_pixel_y"

gene_df.rename(columns={'x': st_x_col, 'y': st_y_col}, inplace=True)

print(
    "Number of transcripts in st data:", len(gene_df),
    "\nNumber of genes in st data", gene_df[gene_col].nunique(),
    "\nNumber of cell types in ref data:", len(markers),
    "\nNumber of unique marker genes in ref data:", len(set(gene for cell_type in markers.values() for genes in cell_type.values() for gene in genes))  
)

# 色情報の読み込み
celltype_color_dict = load_color_info(cfg, dataset_dir, list(markers.keys()))
fig, _ax = plot_color_palette(celltype_color_dict)
plt.show()

整数エンコーディング他

In [ ]:
# 文字列であるcelltype名、gene名を整数エンコーディング
gene_registry = GeneRegistry(
    sc_genes=set(gene for cell_type in markers.values() for genes in cell_type.values() for gene in genes),
    st_genes=gene_df[gene_col].unique().tolist(),
)

cluster_celltype_dict = {'cluster_1': ['NK cells'],
 'cluster_2': ['T lymphocytes'],
 'cluster_3': ['B lymphocytes'],
 'cluster_4': ['Myeloid cells'],
 'cluster_5': ['MAST cells'],
 'cluster_6': ['Endothelial cells'],
 'cluster_7': ['Fibroblasts'],
 'cluster_8': ['Epithelial cells']}

type_registry = build_type_registry(cluster_celltype_dict)
type_registry = append_colors_to_registry(type_registry, celltype_color_dict)

type_system = TypeSystem(type_registry)

valid_ids = type_system.valid_type_ids
unknown_id = type_system.unknown_id
background_id = type_system.background_id
C = type_system.C

# markers を整数エンコーディングする
markers_id = {}

for celltype, marker_dict in markers.items():
    markers_id[type_system.celltype_to_type_id(celltype)] = {}

    pos_list = marker_dict["pos_markers"]
    markers_id[type_system.celltype_to_type_id(celltype)]["pos_gids"] = np.array([gene_registry.gene_to_gid[pos_list[idx]] for idx in range(len(pos_list))])

    neg_list = marker_dict["neg_markers"]
    markers_id[type_system.celltype_to_type_id(celltype)]["neg_gids"] = np.array([gene_registry.gene_to_gid[neg_list[idx]] for idx in range(len(neg_list))])

## step1

In [ ]:
# Step1
quadtree, leaves_df, gene_df_with_node, weights_sparse = run_step1(cfg, gene_df, gene_registry, type_system, type_system.unique_celltypes_list, markers_id)  
fig_qtree, ax_qtree, fig_weights, axes_weights = visualize_step1(cfg, quadtree, leaves_df, gene_df_with_node, weights_sparse, type_system)
plt.show()

In [ ]:
cfg.save_config.enabled = False
cfg.save_config.phase = "step1"
mes = {"description": "Step 1 outputs from xenium human lung cancer dataset"}
save_step1_outputs(dataset_dir, cfg=cfg, ts_data=ts_data, fig_qtree=fig_qtree, fig_weights=fig_weights, gene_df_node=gene_df_with_node, metamesseage=mes)

## step2

In [ ]:
# 実行時間短縮の為、反復回数を減らしている
# cfg.model_config.max_iter = 3

In [ ]:
result_df, gmm_mu_init_dict, histogram_dict = run_step2(cfg, gene_df_with_node, weights_sparse, type_system)
fig_initial, axes_initial, fig_cluster, axes_cluster = visualize_step2(result_df, gmm_mu_init_dict, histogram_dict, valid_ids, type_system, cfg)

plt.tight_layout()
plt.show()

In [ ]:
cfg.save_config.enabled = False
cfg.save_config.phase = "step2"
mes = {"description": "Step 2 outputs from xenium human lung cancer dataset"}
save_step2_outputs(dataset_dir, cfg=cfg, ts_data=ts_data, fig_initial=fig_initial, fig_cluster=fig_cluster, result_df=result_df, metamesseage=mes)

## evaluation

In [ ]:
evaluation_df = result_df.copy() # 名称的に分かりやすくするため
evaluation_dict = evaluate_performance(
    evaluatioin_df=evaluation_df,
    type_system=type_system,
    config={
        'background_id': type_system.background_id,
        'unknown_id': type_system.unknown_id,
        'compare_bg_id': type_system.background_id,
        'proposed_cell_id': "cluster_id",
        'compared_cell_id': None,
        'proposed_celltype': "result_celltype",
        'compared_celltype': None
    })

print_evaluation_summary(evaluation_dict, type_system)

In [ ]:
cfg.save_config.enabled = False
cfg.save_config.phase = "eval"
mes = "free comment"
save_eval_outputs(dataset_dir, cfg=cfg,ts_data=ts_data, evaluation_data=evaluation_dict, metamesseage=mes)